In [8]:
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

/home/eren/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
dataset = load_dataset("Overfit-GM/turkish-toxic-language")

In [11]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    
dataset = dataset.map(tokenize, batched=True)

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "is_toxic"]
)

In [15]:
dataset = dataset["train"].train_test_split(test_size=0.1)

train_ds = dataset["train"]
val_ds = dataset["test"]

In [16]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'target', 'source', 'is_toxic', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 70020
    })
    test: Dataset({
        features: ['text', 'target', 'source', 'is_toxic', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7780
    })
})


In [18]:
class ToxicDataset(Dataset):
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        return {
            "input_ids": item["input_ids"].clone().detach() if torch.is_tensor(item["input_ids"]) else torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": item["attention_mask"].clone().detach() if torch.is_tensor(item["attention_mask"]) else torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["is_toxic"], dtype=torch.float) 
        }

In [19]:
from torch.utils.data import DataLoader

train_loader = DataLoader(ToxicDataset(train_ds), batch_size=16, shuffle=True)
val_loader = DataLoader(ToxicDataset(val_ds), batch_size=32, shuffle=False)


In [25]:
class ToxicBERT(nn.Module):
    def __init__(self):
        super().__init__()

        self.bert = AutoModel.from_pretrained("dbmdz/bert-base-turkish-cased")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls)
        x = self.fc(x)

        return x  # sigmoid yok!

In [29]:
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score


def find_best_threshold(probs, labels, start=0.10, end=0.90, step=0.02):
    probs = np.asarray(probs)
    labels = np.asarray(labels).astype(int)

    best_t = 0.5
    best_f1 = -1.0

    for t in np.arange(start, end + 1e-8, step):
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)

    return best_t, best_f1


def evaluate(model, dataloader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].unsqueeze(1).to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits).squeeze(1)
            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(labels.squeeze(1).detach().cpu().numpy().astype(int).tolist())

    probs_arr = np.asarray(all_probs)
    labels_arr = np.asarray(all_labels).astype(int)
    preds_arr = (probs_arr >= threshold).astype(int)

    precision = precision_score(labels_arr, preds_arr, zero_division=0)
    recall = recall_score(labels_arr, preds_arr, zero_division=0)
    f1 = f1_score(labels_arr, preds_arr, zero_division=0)

    return {
        "loss": total_loss / max(len(dataloader), 1),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "probs": all_probs,
        "labels": labels_arr.tolist(),
        "preds": preds_arr.tolist(),
    }

In [26]:
from transformers import AutoModel
import torch.nn as nn

class ToxicModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("dbmdz/bert-base-turkish-cased")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        x = self.dropout(cls)
        x = self.fc(x)
        return x

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [13]:
from copy import deepcopy
from tqdm import tqdm
from torch.amp import autocast, GradScaler
from transformers import get_linear_schedule_with_warmup

model = ToxicModel().to(device)

train_labels = torch.tensor(
    [float(train_ds[i]["is_toxic"]) for i in range(len(train_ds))],
    dtype=torch.float,
)
num_pos = train_labels.sum().item()
num_neg = len(train_labels) - num_pos
pos_weight = torch.tensor([num_neg / max(num_pos, 1.0)], device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

num_epochs = 5
total_steps = len(train_loader) * num_epochs
num_warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_steps,
)

scaler = GradScaler(enabled=(device.type == "cuda"))

best_val_f1 = 0.0
best_threshold = 0.5
best_state_dict = None
patience = 2
patience_counter = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    val_for_threshold = evaluate(model, val_loader, criterion, device, threshold=best_threshold)
    tuned_threshold, _ = find_best_threshold(val_for_threshold["probs"], val_for_threshold["labels"])
    val_metrics = evaluate(model, val_loader, criterion, device, threshold=tuned_threshold)

    avg_train_loss = total_loss / max(len(train_loader), 1)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    print(f"Val Precision: {val_metrics['precision']:.4f}")
    print(f"Val Recall: {val_metrics['recall']:.4f}")
    print(f"Val F1: {val_metrics['f1']:.4f}")
    print(f"Best Threshold (this epoch): {tuned_threshold:.2f}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_threshold = tuned_threshold
        best_state_dict = deepcopy(model.state_dict())
        patience_counter = 0
        print("Best model updated.")
    else:
        patience_counter += 1
        print(f"No improvement. Early stopping counter: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "threshold": best_threshold,
}
torch.save(checkpoint, "best_toxic_model.pt")
print(f"Training complete. Best Val F1: {best_val_f1:.4f}, threshold: {best_threshold:.2f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3642.14it/s]
BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/5:   0%|          | 0/4377 [00:00<?, ?it/s]/tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or 


Epoch 1/5
Train Loss: 0.1753
Val Loss: 0.0825
Val Precision: 0.9780
Val Recall: 0.9746
Val F1: 0.9763
Best Threshold (this epoch): 0.64
Best model updated.


Epoch 2/5:   0%|          | 0/4377 [00:00<?, ?it/s]/tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)
  0%|          | 0/244 [00:00<?, ?it/s]          /tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)



Epoch 2/5
Train Loss: 0.0635
Val Loss: 0.0949
Val Precision: 0.9784
Val Recall: 0.9786
Val F1: 0.9785
Best Threshold (this epoch): 0.26
Best model updated.


Epoch 3/5:   0%|          | 0/4377 [00:00<?, ?it/s]/tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)
  0%|          | 0/244 [00:00<?, ?it/s]          /tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)



Epoch 3/5
Train Loss: 0.0228
Val Loss: 0.1257
Val Precision: 0.9675
Val Recall: 0.9848
Val F1: 0.9761
Best Threshold (this epoch): 0.72
No improvement. Early stopping counter: 1/2


Epoch 4/5:   0%|          | 0/4377 [00:00<?, ?it/s]/tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)
  0%|          | 0/244 [00:00<?, ?it/s]          /tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)



Epoch 4/5
Train Loss: 0.0093
Val Loss: 0.1386
Val Precision: 0.9770
Val Recall: 0.9828
Val F1: 0.9799
Best Threshold (this epoch): 0.90
Best model updated.


Epoch 5/5:   0%|          | 0/4377 [00:00<?, ?it/s]/tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)
  0%|          | 0/244 [00:00<?, ?it/s]          /tmp/ipykernel_11682/4277196614.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(item["is_toxic"], dtype=torch.float)



Epoch 5/5
Train Loss: 0.0041
Val Loss: 0.1500
Val Precision: 0.9830
Val Recall: 0.9771
Val F1: 0.9800
Best Threshold (this epoch): 0.18
Best model updated.
Training complete. Best Val F1: 0.9800, threshold: 0.18


# TEST

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model = ToxicModel().to(device)
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")


final_metrics = evaluate(model, val_loader, criterion, device, threshold=best_threshold)

print(classification_report(final_metrics["labels"], final_metrics["preds"], digits=4))
print("Confusion matrix:")
print(confusion_matrix(final_metrics["labels"], final_medevicetrics["preds"]))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7602.81it/s]
BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'pos_weight' is not defined

In [15]:
print("Mixed precision (AMP) is already integrated into the main training loop.")

Mixed precision (AMP) is already integrated into the main training loop.


In [5]:
def predict_text(model, tokenizer, text, device, max_length=128, threshold=0.5):
    model.eval()
    tokens = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids = tokens["input_ids"].to(device)
    attention_mask = tokens["attention_mask"].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        prob = torch.sigmoid(logits).item()
        return 1 if prob > threshold else 0

In [40]:
predict_text(model, tokenizer, "NAber", device, threshold=0.18)

1